1.  **Regularization**:

    - Use the `diabetes` dataset from `sklearn.datasets`.
    - Compare the performance (Mean Squared Error) of `LinearRegression`, `Ridge`, and `Lasso` models.
    - Tune the `alpha` parameter for `Ridge` and `Lasso` using `GridSearchCV` with cross-validation to find the optimal regularization strength.

In [237]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline

# Load the diabetes dataset
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target
# print(X)
# print(X.isna().sum())

# Preprocessing for numerical data
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Bundle preprocessing for numerical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, diabetes.feature_names)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

linear_reg_model = LinearRegression()
linear_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('model', linear_reg_model)
                          ])
linear_pipeline.fit(X_train, y_train)

# Generate predictions on the test set
linear_preds = linear_pipeline.predict(X_test)

# Calculate Mean Squared Error (MSE) to quantify prediction accuracy (lower is better)
linear_mse = mean_squared_error(y_test, linear_preds)
print("Linear Regression Mean Squared Error:", linear_mse)

Linear Regression Mean Squared Error: 3424.2593342986925


In [238]:
# Initialize a Ridge regression model
ridge_reg_model_tune = Ridge()

# Define the hyperparameter grid
ridge_param_grid = {
    'alpha': [0.1, 0.2, 0.5, 0.8, 1.0]
}

# Initialize GridSearchCV with the model, parameter grid, cross-validation strategy, and scoring metrics
# here we use MSE score as the metrics
ridge_grid_search = GridSearchCV(estimator=ridge_reg_model_tune, param_grid=ridge_param_grid, cv=5, n_jobs=-1, verbose=2, scoring=('neg_mean_squared_error'))

# Create and evaluate the pipeline
ridge_pipeline_tune = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('grid_search', ridge_grid_search)
                          ])
ridge_pipeline_tune.fit(X_train, y_train)

# check best params
best_params = ridge_grid_search.best_params_
best_score = ridge_grid_search.best_score_

print(f"Best Parameters: {best_params}")
print(f"Best Score: {best_score:.4f}")

Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.5; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ........................................

In [239]:
ridge_grid_search.cv_results_

{'mean_fit_time': array([0.00199709, 0.0022789 , 0.00117936, 0.0011528 , 0.00149593]),
 'std_fit_time': array([0.00188536, 0.00229624, 0.00059244, 0.00027149, 0.00133152]),
 'mean_score_time': array([0.00036106, 0.00042892, 0.00033488, 0.00030985, 0.00036855]),
 'std_score_time': array([7.19638115e-05, 1.24240285e-04, 3.44108636e-05, 2.21755426e-05,
        1.16086435e-04]),
 'param_alpha': masked_array(data=[0.1, 0.2, 0.5, 0.8, 1.0],
              mask=[False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'params': [{'alpha': 0.1},
  {'alpha': 0.2},
  {'alpha': 0.5},
  {'alpha': 0.8},
  {'alpha': 1.0}],
 'split0_test_score': array([-2964.98125508, -2965.31670781, -2966.30499305, -2967.20575873,
        -2967.73988625]),
 'split1_test_score': array([-2655.38697722, -2654.25953748, -2651.36408122, -2649.02237567,
        -2647.68830501]),
 'split2_test_score': array([-3112.26847564, -3112.63389861, -3113.65305979, -3114.55064582,
        -3115.0828607

In [240]:
# Initialize a Ridge regression model with best param
ridge_reg_model = Ridge(alpha=1.0)

ridge_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('model', ridge_reg_model)
                          ])
ridge_pipeline.fit(X_train, y_train)
ridge_preds = ridge_pipeline.predict(X_test)

# Calculate the Mean Squared Error (MSE)
ridge_mse = mean_squared_error(y_test, ridge_preds)
print(f'Ridge MSE: {ridge_mse}')


Ridge MSE: 3430.10676248457


In [241]:
# Initialize a Lasso regression model
lasso_reg_model_tune = Lasso()

# Define the hyperparameter grid
lasso_param_grid = {
    'alpha': [0.1, 0.2, 0.5, 0.8, 1.0]
}

# Initialize GridSearchCV with the model, parameter grid, cross-validation strategy, and scoring metrics
# here we use MSE score as the metrics
lasso_grid_search = GridSearchCV(estimator=lasso_reg_model_tune, param_grid=lasso_param_grid, cv=5, n_jobs=-1, verbose=2, scoring=('neg_mean_squared_error'))

# Create and evaluate the pipeline
lasso_pipeline_tune = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('grid_search', lasso_grid_search)
                          ])
lasso_pipeline_tune.fit(X_train, y_train)

# check best params
best_params = lasso_grid_search.best_params_
best_score = lasso_grid_search.best_score_

print(f"Best Parameters: {best_params}")
print(f"Best Score: {best_score:.4f}")

Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.5; total time=   0.0s
[CV] END ..........................................alpha=0.5; total time=   0.0s
[CV] END ........................................

In [242]:
lasso_grid_search.cv_results_

{'mean_fit_time': array([0.00112343, 0.0008183 , 0.00149622, 0.00171714, 0.00126429]),
 'std_fit_time': array([3.00393369e-04, 3.68750849e-05, 1.25596132e-03, 1.50742419e-03,
        9.32013929e-04]),
 'mean_score_time': array([0.00033703, 0.00029597, 0.00029111, 0.00058641, 0.00034819]),
 'std_score_time': array([4.05456293e-05, 1.44053760e-05, 1.18702766e-05, 4.77546322e-04,
        7.67836057e-05]),
 'param_alpha': masked_array(data=[0.1, 0.2, 0.5, 0.8, 1.0],
              mask=[False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'params': [{'alpha': 0.1},
  {'alpha': 0.2},
  {'alpha': 0.5},
  {'alpha': 0.8},
  {'alpha': 1.0}],
 'split0_test_score': array([-2973.20974072, -2980.04253923, -2979.89528004, -2977.28224238,
        -2975.08400769]),
 'split1_test_score': array([-2647.77310833, -2641.37577522, -2636.5927629 , -2638.25767211,
        -2640.39036183]),
 'split2_test_score': array([-3113.45488541, -3113.83053267, -3100.06435166, -3100.304

In [243]:
# Initialize a Lasso regression model with best param
lasso_reg_model = Lasso(alpha=0.5)

lasso_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('model', lasso_reg_model)
                          ])
lasso_pipeline.fit(X_train, y_train)

# Generate predictions on the test set using the trained Lasso model
lasso_preds = lasso_pipeline.predict(X_test)

# Calculate the Mean Squared Error (MSE)
lasso_mse = mean_squared_error(y_test, lasso_preds)
print(f'Lasso MSE: {lasso_mse}')


Lasso MSE: 3431.8668019729053


In [244]:
print(f"LinearRegression MSE:", linear_mse, "\nRidge MSE:", ridge_mse, "\nLasso MSE:", lasso_mse)

LinearRegression MSE: 3424.2593342986925 
Ridge MSE: 3430.10676248457 
Lasso MSE: 3431.8668019729053


2.  **Ensemble Methods**:

    - Use the `breast_cancer` dataset from `sklearn.datasets`.
    - Compare the performance (F1 Score and AUC) of `DecisionTreeClassifier`, `RandomForestClassifier`, and `GradientBoostingClassifier`.
    - Tune the hyperparameters of each classifier using `GridSearchCV` with cross-validation.

In [245]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, auc, roc_curve

# Load the breast cancer dataset
breast_cancer = load_breast_cancer(as_frame=True)
X = breast_cancer.data
y = breast_cancer.target
# print(X)
# print(X.isna().sum())

# Preprocessing for numerical data
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Bundle preprocessing for numerical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, breast_cancer.feature_names)
    ])

decision_tree_model_tune = DecisionTreeClassifier(criterion='gini', random_state=0)
decision_tree_param_grid = {
    'max_depth': [None, 2, 4, 6, 8, 10],
    'min_samples_split': [2, 4, 6, 8],
    'min_samples_leaf': [2, 4, 6, 8],
}

# Use f1 and auc score as the metrics
decision_tree_grid_search = GridSearchCV(estimator=decision_tree_model_tune, param_grid=decision_tree_param_grid, cv=5, n_jobs=-1, verbose=2, scoring=('f1','roc_auc'), refit=False)

decision_tree_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('grid_search', decision_tree_grid_search)
                          ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
decision_tree_pipeline.fit(X_train, y_train)


Fitting 5 folds for each of 96 candidates, totalling 480 fits
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=2; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=2; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=2; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=4; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=2; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=4; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=4; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=2; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=4; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=6; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=4; total time=   0.0s
[CV

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  array(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'tex...
       'worst compactness', 'worst concavity', 'worst concave points',
       'worst symmetry', 'worst fractal dimension'], dtype='<U23'))])),
                ('grid_search',
                 GridSearchCV(cv=5,
                              estimator=DecisionTreeClassifier(random_state=0),
                              n_jobs=-1,
                              param_grid={'max_depth': [None, 2, 4, 6, 8, 10],
                                          'min_samples_leaf': [2, 4, 6, 8],
                                          'min_samples_split': [2, 4, 6, 8]},
                              refit=False, scoring=('f1', 'roc_auc'),
                              verbose=2))])

In [246]:
# Inspect results
import pandas as pd
results_df = pd.DataFrame(decision_tree_grid_search.cv_results_)

# Example: Get best params by accuracy
best_idx_auc = results_df['mean_test_roc_auc'].idxmax()
best_params_auc = results_df.loc[best_idx_auc, 'params']

# Example: Get best params by F1
best_idx_f1 = results_df['mean_test_f1'].idxmax()
best_params_f1 = results_df.loc[best_idx_f1, 'params']

print("Best params by F1:", best_params_f1)
print("Best params by AUC:", best_params_auc)

Best params by F1: {'max_depth': None, 'min_samples_leaf': 8, 'min_samples_split': 2}
Best params by AUC: {'max_depth': None, 'min_samples_leaf': 8, 'min_samples_split': 2}


In [247]:
list(zip(decision_tree_grid_search.cv_results_['params'],decision_tree_grid_search.cv_results_['mean_test_f1'], decision_tree_grid_search.cv_results_['mean_test_roc_auc']))

[({'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2},
  0.9393730786245083,
  0.9265412748171368),
 ({'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 4},
  0.9393730786245083,
  0.9265412748171368),
 ({'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 6},
  0.9374567169077445,
  0.9266980146290491),
 ({'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 8},
  0.9381902456601308,
  0.9331765935214211),
 ({'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 2},
  0.9437628542207227,
  0.9464994775339604),
 ({'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 4},
  0.9437628542207227,
  0.9464994775339604),
 ({'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 6},
  0.9437628542207227,
  0.9464994775339604),
 ({'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 8},
  0.9437628542207227,
  0.9464994775339604),
 ({'max_depth': None, 'min_samples_leaf': 6, 'min_samples_split': 2},
  0.939058

In [248]:
# Initialize a decision tree model with default
decision_tree_model = DecisionTreeClassifier(criterion='gini', random_state=0)
decision_tree_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('model', decision_tree_model)
                          ])
decision_tree_pipeline.fit(X_train, y_train)
decision_tree_preds = decision_tree_pipeline.predict(X_test)

f1 = f1_score(y_test, decision_tree_preds)
fpr, tpr, thresholds = roc_curve(y_test, decision_tree_pipeline.predict_proba(X_test)[:,1])
roc_auc = auc(fpr, tpr)

print(f'F1 Score: {f1:.2f}')
print(f"AUC: {roc_auc:.2f}")


F1 Score: 0.92
AUC: 0.92


In [249]:
# Initialize a decision tree model with best params
decision_tree_model_final = DecisionTreeClassifier(criterion='gini', random_state=0, max_depth=None, min_samples_split=2, min_samples_leaf=8)
decision_tree_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('model', decision_tree_model_final)
                          ])
decision_tree_pipeline.fit(X_train, y_train)
decision_tree_preds = decision_tree_pipeline.predict(X_test)

f1 = f1_score(y_test, decision_tree_preds)
fpr, tpr, thresholds = roc_curve(y_test, decision_tree_pipeline.predict_proba(X_test)[:,1])
roc_auc = auc(fpr, tpr)

print(f'F1 Score: {f1:.2f}')
print(f"AUC: {roc_auc:.2f}")


F1 Score: 0.95
AUC: 0.99
